## La Haute Borne SCADA 功率曲线清洗（bin + MAD）功能演示

数据：法国 La Haute Borne 风场 2017–2020 四台 **Senvion MM82（2.05 MW）**，列 `Ws_avg`（机舱风速 m/s）、`P_avg`（有功功率 kW）。

`PowerCurveFiltering` 一次只处理**单台风机**，多台由调用方 `groupby` 后循环调用。

`cut_in_speed` / `rated_wind_speed` / `rated_power` 是厂商机型物理参数，**必填**。厂商参数未知时，可先调用 `estimate_machine_parameters` 从 SCADA 反推（10 分钟统计口径的“软”值，额定风速比厂商标称约低 1 m/s），再显式传入。


In [1]:
import sys
sys.path.append('../')
from wtclean import PowerCurveFiltering, estimate_machine_parameters

In [2]:
import pandas as pd
df = pd.read_csv('./datasets/la-haute-borne-data-2017-2020.zip', sep=';')
df['Ws_avg'] = pd.to_numeric(df['Ws_avg'], errors='coerce')
df['P_avg']  = pd.to_numeric(df['P_avg'],  errors='coerce')
df.head()

,Wind_turbine_name,Date_time,Ba_avg,Ba_min,Ba_max,Ba_std,Rt_avg,Rt_min,Rt_max,Rt_std,...,Pas_max,Pas_std,Wa_c_avg,Wa_c_min,Wa_c_max,Wa_c_std,Na_c_avg,Na_c_min,Na_c_max,Na_c_std
0,R80721,2017-02-08T08:00:00+01:00,44.990002,44.990002,44.990002,0.0,14.0,14.0,14.0,0.0,...,NaN,NaN,358.04999,NaN,NaN,NaN,358.04999,NaN,NaN,NaN
1,R80721,2017-01-26T02:40:00+01:00,-1.000000,-1.000000,-1.000000,0.0,10.0,10.0,10.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R80721,2017-01-26T13:50:00+01:00,-1.000000,-1.000000,-1.000000,0.0,10.0,10.0,10.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,R80721,2017-01-26T15:00:00+01:00,-1.000000,-1.000000,-1.000000,0.0,10.0,10.0,10.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,R80721,2017-02-18T01:10:00+01:00,44.990002,44.990002,44.990002,0.0,17.0,17.0,17.0,0.0,...,NaN,NaN,7.99000,NaN,NaN,NaN,7.99000,NaN,NaN,NaN


### 单台清洗：显式传入厂商物理参数（Senvion MM82）

`cut_in_speed=3.5`、`rated_wind_speed=14.5`、`rated_power=2050`（上方已先打印反推值作参考）。


In [ ]:
g = df[df['Wind_turbine_name'] == 'R80721']

output = estimate_machine_parameters(
    df=g, windspeed_label='Ws_avg', power_label='P_avg',
)
print(output)

pc = PowerCurveFiltering(
    windspeed_label='Ws_avg',   # 风速列
    power_label='P_avg',        # 功率列
    df=g,                       # 单台风机
    cut_in_speed=3.5,           # 切入风速（厂商）
    rated_wind_speed=14.5,      # 额定风速（厂商）
    rated_power=2050,           # 额定功率 kW（厂商）
    low_power_ratio=0.9,
    bin_interval=0.5,
    z_coeff=2.5,
    filter_cycle=3,
    return_fig=True,            # 存一张功率曲线清洗图
    image_path='./images/R80721.png',

)
normal_df, abnormal_df = pc.process()

print(f'total        = {len(g)}')
print(f'normal       = {len(normal_df)}')
print(f'abnormal     = {len(abnormal_df)}')
print(f'abnormal 比例 = {len(abnormal_df) / len(g):.2%}')

{'cut_in_speed': 3.79, 'rated_wind_speed': 12.72, 'rated_power': 1985.3}
iteration 1: removed 3864 (7.21%), remaining 49724
iteration 2: removed 1655 (3.33%), remaining 48069
iteration 3: removed 904 (1.88%), remaining 47165
iteration 4: removed 489 (1.04%), remaining 46676
iteration 5: removed 230 (0.49%), remaining 46446
total        = 54433
normal       = 46446
abnormal     = 7987
abnormal 比例 = 14.67%
